In [2]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   --------- ------------------------------ 1.8/8.0 MB 14.4 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.0 MB 15.0 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 16.5 MB/s  0:00:00

   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   ------

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss, roc_auc_score
import matplotlib.pyplot as plt

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [7]:
def load_and_split_data(filepath, split_type='chronological', test_size=0.2):
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    # Assuming data structure: {'features': [...], 'targets': [...]}
    X = np.array(data['features'])
    y = np.array(data['targets'])
    
    if split_type == 'chronological':
        # Essential for xT to prevent data leakage from future match states
        split_idx = int(len(X) * (1 - test_size))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        
    return X_train, X_test, y_train, y_test


In [5]:
class ModelEvaluator:
    def __init__(self, model_path, model_class, model_params):
        self.model = model_class(**model_params).to(device)
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.eval()

    def evaluate(self, test_loader):
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = self.model(inputs)
                
                # If your model outputs logits, apply sigmoid for probability
                probs = torch.sigmoid(outputs) if hasattr(self.model, 'is_logit_output') else outputs
                
                all_preds.extend(probs.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
        
        return np.array(all_preds), np.array(all_targets)

    def calculate_metrics(self, preds, targets):
        # Brier Score: Mean Squared Error of probabilities
        bs = brier_score_loss(targets, preds)
        # AUC: Discrimination ability
        auc = roc_auc_score(targets, preds)
        
        print(f"--- Evaluation Metrics ---")
        print(f"Brier Score: {bs:.5f}")
        print(f"ROC-AUC:     {auc:.5f}")
        
        return {"brier": bs, "auc": auc}

In [6]:
def plot_calibration_curve(preds, targets, bins=10):
    bin_edges = np.linspace(0, 1, bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    true_proportions = []
    pred_proportions = []
    
    for i in range(bins):
        mask = (preds >= bin_edges[i]) & (preds < bin_edges[i+1])
        if np.any(mask):
            true_proportions.append(np.mean(targets[mask]))
            pred_proportions.append(np.mean(preds[mask]))
            
    plt.figure(figsize=(8, 6))
    plt.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
    plt.plot(pred_proportions, true_proportions, 's-', label='TransGoalNet')
    plt.ylabel('Fraction of Positives (Actual Goals)')
    plt.xlabel('Mean Predicted xT')
    plt.title('Reliability Diagram (Calibration Curve)')
    plt.legend()
    plt.grid(True)
    plt.show()

In [12]:
# Load data
X_train, X_test, y_train, y_test = load_and_split_data(filepath="D:/MSC DS ABDN/PROJ_The science of football - Jan 2026/CODE/Vibe/football_analytics_framework/data/all_competitions_events.pkl")

KeyError: 'features'